# 06 — Групповой screening моделей

Этот notebook сравнивает одну группу моделей на **замороженном feature set** и одинаковых CV-folds. Он не меняет EDA-признаки и не выполняет tuning.

Настройки групп и гиперпараметров: `src/ml_project/model_screening_config.py`. Результат сохраняется в отдельную карточку `model-screening/MS-xxx ...md`.

## Правила этапа

1. Feature reference уже принят и не меняется внутри screening.
2. Один `MS-ID` соответствует одной группе и одному набору стартовых параметров.
3. Первое место — только кандидат в shortlist, а не доказанный финальный победитель.
4. Новая группа получает новый `MS-ID`, карточку, `run_name` и тот же validation contract.

In [ ]:
from pathlib import Path
import importlib
import sys

from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / 'README.md').exists() and (candidate / 'src').exists()
)
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project import DataCatalog
import ml_project.config as project_config
import ml_project.baseline_config as baseline_config
import ml_project.modeling as modeling_tools
import ml_project.model_screening as screening_tools
import ml_project.model_screening_config as screening_config

print(f'Корень проекта: {PROJECT_ROOT}')

## 1. Перечитать конфиг и увидеть параметры до обучения

Если здесь не та группа, feature reference или параметры — остановитесь и исправьте config.

In [ ]:
project_config = importlib.reload(project_config)
baseline_config = importlib.reload(baseline_config)
modeling_tools = importlib.reload(modeling_tools)
screening_tools = importlib.reload(screening_tools)
screening_config = importlib.reload(screening_config)

DATASETS = project_config.DATASETS
FEATURE_GROUPS = project_config.FEATURE_GROUPS
KEY = project_config.KEY
RAW_DIR = project_config.RAW_DIR
TARGET = project_config.TARGET
TRAIN_DATASET = project_config.TRAIN_DATASET
initial_settings = baseline_config.BASELINE
SCREENING = screening_config.SCREENING

modeling_tools.validate_modeling_settings(initial_settings)
screening_tools.validate_screening_settings(SCREENING)
display(screening_tools.screening_settings_report(SCREENING))
display(screening_tools.configured_models_report(SCREENING))

## 2. Загрузить данные и восстановить feature champion

Pipeline принятого experiment пересобирается из кода; старые CSV и fitted-модели не используются.

In [ ]:
catalog = DataCatalog(PROJECT_ROOT, RAW_DIR, DATASETS)
catalog.validate()
train = catalog.load(TRAIN_DATASET)
train_file = catalog.file_report().set_index('dataset').loc[TRAIN_DATASET]
dataset_version = str(train_file['sha256'])

context = screening_tools.prepare_screening_context(
    PROJECT_ROOT,
    train,
    FEATURE_GROUPS,
    target=TARGET,
    key=KEY,
    initial_settings=initial_settings,
    feature_reference_module=SCREENING.feature_reference_module,
)
print('Feature reference:', context.feature_reference_id)
print('Feature module SHA-256:', context.feature_reference_sha256 or 'baseline')
print('Матрица до fold-safe pipeline:', context.data.X.shape)
display(context.plan.to_frame())

## 3. Собрать одну группу

`feature_champion` остаётся точной прежней pipeline. Кандидаты получают preprocessing-профиль группы и параметры из config.

In [ ]:
built = screening_tools.build_model_group(context, SCREENING)
print('Модели:', ', '.join(built.models))
display(built.parameters)

## 4. Выполнить screening на одинаковых folds

Положительный `improvement_vs_reference` всегда означает улучшение, включая minimize-метрики. Wins/losses считаются попарно на тех же folds.

In [ ]:
result = screening_tools.run_model_screening(PROJECT_ROOT, built, SCREENING)
display(
    result.leaderboard[[
        'rank', 'model', 'metric', 'mean ± std',
        'improvement_vs_reference', 'fold_wins', 'fold_ties',
        'fold_losses', 'shortlisted',
    ]].round(4)
)
print('Shortlist для следующего этапа:', result.shortlist)

## 5. Универсальная диагностика группы

Здесь нет повторного fit: используются fitted estimators и validation-индексы уже выполненных folds.

In [ ]:
display(result.prediction_comparison.round(4))
display(
    result.paired_deltas[
        result.paired_deltas['metric_key'].eq('primary')
    ][['model', 'fold', 'candidate_value', 'reference_value', 'improvement']].round(4)
)
diagnostic_model = SCREENING.diagnostic_model_id or result.shortlist[0]
diagnostic_importance = result.feature_importance[
    result.feature_importance['model'].eq(diagnostic_model)
].head(20)
if diagnostic_importance.empty:
    print(f'{diagnostic_model}: у estimator нет native importance/coef_.')
else:
    display(diagnostic_importance.round(4))

## 6. Построить графики

При финальном сохранении они попадут в `assets/model-screening/<MS-ID>/` и автоматически встроятся в карточку.

In [ ]:
figures = screening_tools.build_screening_figures(result, SCREENING)
for name, figure in figures.items():
    print(name)
    display(figure)

## 7. Явно сохранить официальный screening

Эта последняя ячейка — единственное место записи карточки, registry, локальных таблиц и Git-tracked PNG.

In [ ]:
saved = screening_tools.save_model_screening(
    PROJECT_ROOT,
    result,
    SCREENING,
    dataset_version=dataset_version,
    figures=figures,
    config_path=Path(screening_config.__file__),
)
print('Локальные таблицы:', saved.run_dir.relative_to(PROJECT_ROOT))
print('Карточка:', saved.note_path.relative_to(PROJECT_ROOT))
print('Git-tracked графики:', saved.figure_dir.relative_to(PROJECT_ROOT))
print('Registry:', saved.registry_path.relative_to(PROJECT_ROOT))

## 8. Что делать после группы

1. Заполните интерпретацию и решение в созданной карточке.
2. Для следующей группы создайте новый `MS-ID`, note и `run_name`, затем смените `active_group`.
3. После всех групп оставьте 2–3 разных семейства в общем shortlist.
4. Только shortlist переводите в coarse tuning; старые feature-эксперименты массово не повторяйте.